# Notebook Docente 01 — TOA (L1C) vs. BOA (L2A)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miguepoloc/teledeteccion/blob/main/sesiones/sesion-02/docente/colab/01_toa_vs_boa.ipynb)

## Teledetección — Maestría en Ingeniería, Universidad del Magdalena
**Sesión 2 · Sábado 18 de julio de 2026 · Bloque 2**

---

**Este notebook es para que TÚ (el docente) lo proyectes en pantalla.** Versión
en Python/Colab del script `docente/gee/01_toa_vs_boa.js`.

### Qué vas a mostrar
La misma escena Sentinel-2, en dos versiones: L1C (`COPERNICUS/S2_HARMONIZED`,
sin corrección atmosférica) y L2A (`COPERNICUS/S2_SR_HARMONIZED`, corregida).
Comparas la banda azul (más velada en L1C) y el valor de NDVI en el mismo
punto (más bajo en L1C) — la analogía del "vaso de agua con lápiz" del
Bloque 2 se vuelve un número real.

In [3]:
!pip install geemap --quiet
print("✓ Instalación completada")

✓ Instalación completada


In [4]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='teledeteccion-miguepoloc')

print("✓ Google Earth Engine inicializado correctamente")

✓ Google Earth Engine inicializado correctamente


## Paso 1 — Cargar L1C y L2A de la misma zona y fechas

In [5]:
zona_estudio = ee.Geometry.Rectangle([-74.2, 10.5, -73.8, 11.0])

imagen_l1c = (
    ee.ImageCollection('COPERNICUS/S2_HARMONIZED')  # L1C — Top of Atmosphere
    .filterBounds(zona_estudio)
    .filterDate('2024-01-01', '2024-03-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .median()
    .clip(zona_estudio)
)

imagen_l2a = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')  # L2A — Surface Reflectance
    .filterBounds(zona_estudio)
    .filterDate('2024-01-01', '2024-03-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .median()
    .clip(zona_estudio)
)

print("✓ Ambas imágenes listas")

✓ Ambas imágenes listas


## Paso 2 — NDVI en ambas y mapa comparativo

In [6]:
ndvi_l1c = imagen_l1c.normalizedDifference(['B8', 'B4']).rename('NDVI_L1C')
ndvi_l2a = imagen_l2a.normalizedDifference(['B8', 'B4']).rename('NDVI_L2A')
diferencia_ndvi = ndvi_l2a.subtract(ndvi_l1c).rename('Diferencia_NDVI')

paleta_ndvi = ['#d73027', '#f46d43', '#fdae61', '#ffffbf', '#a6d96a', '#1a9641']
vis_color_natural = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000, 'gamma': 1.4}

m = geemap.Map()
m.centerObject(zona_estudio, 12)
m.addLayer(imagen_l1c, vis_color_natural, '1 - L1C Color Natural (SIN corregir)')
m.addLayer(imagen_l2a, vis_color_natural, '2 - L2A Color Natural (corregida)', False)
m.addLayer(ndvi_l1c, {'min': -0.1, 'max': 0.9, 'palette': paleta_ndvi}, '3 - NDVI sobre L1C', False)
m.addLayer(ndvi_l2a, {'min': -0.1, 'max': 0.9, 'palette': paleta_ndvi}, '4 - NDVI sobre L2A', False)
m.addLayer(diferencia_ndvi, {'min': -0.2, 'max': 0.2, 'palette': ['#d73027', '#ffffff', '#1a9641']},
           '5 - Diferencia NDVI (L2A menos L1C)', False)
m.addLayerControl()
m

Map(center=[10.749994930545569, -73.99999999999935], controls=(WidgetControl(options=['position', 'transparent…

## Paso 3 — Comparar valores exactos en un punto de vegetación y uno de agua

In [7]:
def comparar_punto(nombre, punto):
    b2_l1c = imagen_l1c.select('B2').reduceRegion(ee.Reducer.mean(), punto.buffer(15), 10).get('B2').getInfo()
    b2_l2a = imagen_l2a.select('B2').reduceRegion(ee.Reducer.mean(), punto.buffer(15), 10).get('B2').getInfo()
    ndvi1  = ndvi_l1c.reduceRegion(ee.Reducer.mean(), punto.buffer(15), 10).get('NDVI_L1C').getInfo()
    ndvi2  = ndvi_l2a.reduceRegion(ee.Reducer.mean(), punto.buffer(15), 10).get('NDVI_L2A').getInfo()
    print(f"=== {nombre} ===")
    print(f"  B2 (azul)  L1C: {b2_l1c}   L2A: {b2_l2a}")
    print(f"  NDVI       L1C: {ndvi1:.4f}   L2A: {ndvi2:.4f}")
    print()

punto_vegetacion = ee.Geometry.Point([-74.02, 10.75])
punto_agua = ee.Geometry.Point([-74.15, 10.85])

comparar_punto('Punto de vegetación (cacaotal)', punto_vegetacion)
comparar_punto('Punto de agua (Ciénaga Grande)', punto_agua)

=== Punto de vegetación (cacaotal) ===
  B2 (azul)  L1C: 889.372722252899   L2A: 257.4215902816124
  NDVI       L1C: 0.7103   L2A: 0.8366

=== Punto de agua (Ciénaga Grande) ===
  B2 (azul)  L1C: 1093.5022002200221   L2A: 358.6331133113312
  NDVI       L1C: 0.5631   L2A: 0.6947



## Guion para la clase

- **¿Por qué B2 (azul) es más alto en L1C que en L2A?** La dispersión de Rayleigh
  de la atmósfera añade brillo extra en longitudes de onda cortas que el L2A ya
  elimina.
- **¿Por qué el NDVI de L1C suele ser más bajo?** La atmósfera reduce el
  contraste rojo/NIR que el NDVI necesita — "aplana" la señal.
- **Regla del curso:** SIEMPRE usar L2A (`COPERNICUS/S2_SR_HARMONIZED`) para
  cualquier análisis cuantitativo o comparación temporal.